In [8]:
import os
from dotenv import load_dotenv

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_community.embeddings import JinaEmbeddings
from langchain_community.vectorstores import FAISS  

from langchain_groq import ChatGroq
from langchain.agents import create_agent


C:\Users\shashavali\AppData\Local\Temp\ipykernel_4844\1398374472.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [9]:
load_dotenv()

True

In [10]:
groq_key = os.getenv("GROQ_API_KEY")

jina_key = os.getenv("JINA_API_KEY")

print("ENV VAR lOADED")


ENV VAR lOADED


Loading our data 


In [11]:
DATA_FILE_PATH = os.path.join("data" , "hr_policy.txt")

DATA INGESTION

In [12]:
loader = TextLoader(DATA_FILE_PATH, encoding="utf-8")

documents = loader.load()
print("DATA LOADED")
print("="*40)
print(documents)

DATA LOADED
[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)\n\n1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.\n\n2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM to 4 PM.\n\n3. PROBATION PERIOD\nAll new employees undergo a probation period of 3 months from their date of joining.\nDu

#### LANGCHAIN DOCUMENT

Langchain processes everything in form of documents 

In [13]:
len(documents)

1

In [14]:
documents[0].metadata

{'source': 'data\\hr_policy.txt'}

In [15]:
print("Total number of documents loaded:", len(documents[0].page_content))

Total number of documents loaded: 2598


SPLITTING OUR DATA 

In [16]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(chunks)

[Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='COMPANY HR POLICY HANDBOOK\nAcme Corp - Employee Handbook (Demo Document)'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='1. LEAVE POLICY\nAll full-time employees are entitled to 20 days of paid annual leave per calendar year.\nLeave requests must be submitted through the HR portal at least 5 working days in advance.\nUnused annual leave can be carried forward to the next year, up to a maximum of 5 days.\nSick leave is separate from annual leave, and employees get 10 paid sick days per year.\nA medical certificate is required for sick leave longer than 2 consecutive days.'), Document(metadata={'source': 'data\\hr_policy.txt'}, page_content='2. WORK FROM HOME POLICY\nEmployees may work from home up to 2 days per week, subject to manager approval.\nFully remote work arrangements require written approval from the department head.\nEmployees working from home must be reachable during core hours: 10 AM

In [17]:
len(chunks)

9

NOW EACH SPLITTED CHUNK IS A DOCUMENT 


In [18]:
print(chunks[8])

page_content='8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.' metadata={'source': 'data\\hr_policy.txt'}


In [19]:
print(chunks[8])

page_content='8. EXIT POLICY
Upon resignation or termination, employees must complete a clearance process involving
IT, Finance, and HR departments before their last working day.
Full and final settlement, including any pending reimbursements and leave encashment,
is processed within 45 days of the last working day.' metadata={'source': 'data\\hr_policy.txt'}


EMBEDD OUR DATA 

In [20]:
embeddings_model = JinaEmbeddings(model_name="jina-embeddings-v2-base-en")

print("Embedding model loaded", embeddings_model.model_name)


Embedding model loaded jina-embeddings-v2-base-en


### STORING DATA IN VECTOR DB

In [21]:
vector_store = FAISS.from_documents(chunks, embeddings_model)
print("CHUNKS ARE STORED IN VECTOR DB is", vector_store.index.ntotal)

CHUNKS ARE STORED IN VECTOR DB is 9


WE NEVER STORED IT 


In [22]:
test_query = "How many sick leaves employees get"

## similarity search

top_matches = vector_store.similarity_search(test_query, k=3)

print(f" query: {test_query}")

for i, match in enumerate(top_matches, start=1):
    print(f"Match {i}:")
    print(match.page_content)
    print("="*40)

 query: How many sick leaves employees get
Match 1:
1. LEAVE POLICY
All full-time employees are entitled to 20 days of paid annual leave per calendar year.
Leave requests must be submitted through the HR portal at least 5 working days in advance.
Unused annual leave can be carried forward to the next year, up to a maximum of 5 days.
Sick leave is separate from annual leave, and employees get 10 paid sick days per year.
A medical certificate is required for sick leave longer than 2 consecutive days.
Match 2:
7. HOLIDAYS
The company observes 12 public holidays every year, as per the official holiday calendar
published by HR at the start of each year.
Employees working on a public holiday are eligible for compensatory leave.
Match 3:
3. PROBATION PERIOD
All new employees undergo a probation period of 3 months from their date of joining.
During probation, employees are not eligible for paid leave, but may take unpaid leave
in case of emergencies, subject to manager approval.
Performance is

### DATA RETRIVAL 


In [23]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0, # creativity of the model's
)

llm.model_name

'openai/gpt-oss-120b'

In [24]:
test_response = llm.invoke("Hey is learning rag hard? answer in a single line.")

In [25]:
test_response.content

'Learning RAG can be challenging at first, but with clear resources and practice, it becomes manageable.'

AI AGENT

In [30]:
from langchain_core.tools import tool

@tool
def search_hr_policy(query: str) -> str:
    """Search the HR policy document for relevant passages."""
    matches = vector_store.similarity_search(query, k=3)
    if not matches:
        return "No relevant HR policy information was found."

    return "\n\n".join(match.page_content for match in matches)

hr_assistant = create_agent(
    model=llm,
    tools=[search_hr_policy],
    system_prompt="""
    You are a friendly HR assistant working for Acme Corp.
    Always use the search_hr_policy tool to look up facts before answering.
    If the answer is not present in the search results, say you do not know rather than guessing.
    """
)

print("HR assistant agent is ready to answer questions!")

HR assistant agent is ready to answer questions!


In [37]:
def ask_hr_assistant(question: str) -> str:
    """Send a question to the RAG agent and print a nicely formatted answer."""
    print("=" * 60)
    print("QUESTION:", question)
    print("-" * 60)

    response = hr_assistant.invoke({"messages": [{"role": "user", "content": question}]})
    answer = response["messages"][-1].content

    print("ANSWER:", answer)
    print("=" * 60)
    print()
    return answer

In [38]:
response = hr_assistant.invoke(
    {
        "messages":[
            {
                "role":"user",
                "content": "tell me which org you work for"
            }
        ]
    }
)

In [39]:
response

{'messages': [HumanMessage(content='tell me which org you work for', additional_kwargs={}, response_metadata={}, id='5b1dbf8a-20b3-4010-8781-5dbb08bbccf6'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'The user asks: "tell me which org you work for". The assistant is a friendly HR assistant working for Acme Corp. According to developer instructions, we must always use the search_hr_policy tool to look up facts before answering. The question is about which organization the assistant works for. That is presumably in policy? Might not be in HR policy. But we need to use the tool to search. Let\'s search for "Acme Corp" or "organization".', 'tool_calls': [{'id': 'fc_b0d0de53-e9d0-4f9c-9e44-61c19a5d6ff3', 'function': {'arguments': '{"query":"Acme Corp"}', 'name': 'search_hr_policy'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 124, 'prompt_tokens': 183, 'total_tokens': 307, 'completion_time': 0.268172783, 'completion_tokens_details': {'

In [40]:
response["messages"][-1].content



'I’m an HR assistant for **Acme\u202fCorp**【search_hr_policy query=Acme Corp】.'